## Classification des cellules — EXPLORATION vs PRODUCTION

| # | Contenu | Statut | Raison |
|---|---------|--------|--------|
| 2 | Chargement + export CSV | **PRODUCTION** → `load_data()` | Logique de chargement réutilisable |
| 3 | Imports pandas/numpy/plotly | **PRODUCTION** (partiel) | pd/np gardés ; plotly = exploration |
| 4 | Lecture CSV + conversion kWh | **PRODUCTION** → `load_data()` | Cœur du pipeline |
| 5 | shape / period / memory | EXPLORATION | Audit one-shot, non reproductible |
| 6 | Régularité grille 15min | EXPLORATION | Vérification diagnostique |
| 7–8 | Analyse DST (horloge locale) | EXPLORATION | Insight utile, pas automatisable |
| 9 | Répartition 370 clients (bar chart) | EXPLORATION | Visualisation |
| 10–11 | Zéros / rollout des compteurs | EXPLORATION | Analyse one-shot |
| 12–13 | Saisonnalité annuelle | EXPLORATION | Visualisation |
| 14–15 | Profil jour / semaine | EXPLORATION | Visualisation |
| 16–17 | Dips décembre + jours fériés | EXPLORATION | Analyse diagnostique |
| 18 | Autocorrélation | EXPLORATION (→ justifie les lags) | Oriente le choix des features |
| 19–20 | Construction features + corrélations | **PRODUCTION** → `build_features()` | Feature engineering réutilisable |
| 21 | Split + LinearRegression | **PRODUCTION** → `split_chronological()` + `train_ridge()` | Split et modèle de référence |
| 22–24 | Essais d'alphas / features | EXPLORATION | Expérimentation, jetable |
| 25 | Rediagnostic RMSE/MAE | **PRODUCTION** → `evaluate()` | Métriques officielles |
| 26–28 | Diagnostic résidus / heure | EXPLORATION | Visualisations post-training |
| 29 | Erreur par bucket / client | EXPLORATION | Analyse de décomposition |
| 30 | Comparaison overfitting (arbre) | EXPLORATION | Expérimentation jetable |
| 31 | Courbe de régularisation | EXPLORATION | Justifie le choix alpha=1 |
| 32 | `pickle.dump(m, ...)` | **PRODUCTION** → `save_model()` | Persistance du modèle |

**Obstacles à la modularité / testabilité / reproductibilité relevés :**
- `features` réécrit 3 fois (cellules 21, 24, 25) → état partagé silencieux
- `m` utilisé en fin de notebook sans savoir quel modèle il contient (cellule 32)
- Split par `iloc` (positif) mais non garanti chronologique si l'index n'est pas trié
- Pas de `requirements.txt` / `pyproject.toml` → dépendances implicites
- Visualisations Plotly non reproductibles (données en mémoire du browser)
- `!pip install` absents ici mais fréquents dans les notebooks d'exploration

**Choix industrialisés :**
- **Features** : `lag_1d`, `lag_7d`, `lag_30d`, `rolling_mean_30d` (meilleur ratio corrélation/coût en données)
- **Modèle** : `Ridge(alpha=1.0)` — régularise les lags colinéaires, déterministe, reproductible

# Electricity consumption forecasting - exploration

Working notes on the UCI ElectricityLoadDiagrams dataset (370 clients, 15-minute steps,
2011-2014). Goal: understand the data well enough to decide what a forecasting model should
look like, then try a first model.

Quick and dirty: cells get re-run, variables get overwritten, and whatever looks best at the
end gets pickled.

In [1]:
import urllib.request
import zipfile
from pathlib import Path

import pandas as pd

# The raw file is fetched once by scripts/dataset.py and kept in shared/dataset/.
# Flip DOWNLOAD to True to pull it from UCI instead - the only line here that uses the network.
DOWNLOAD = False
url = (
    "https://archive.ics.uci.edu/static/public/321/electricityloaddiagrams20112014.zip"
)
cache = next(
    parent / "shared" / "dataset" / "LD2011_2014.txt"
    for parent in Path.cwd().parents
    if (parent / "shared" / "dataset").is_dir()
)

if DOWNLOAD:
    filehandle, _ = urllib.request.urlretrieve(url)
    zip_file_object = zipfile.ZipFile(filehandle, "r")
    first_file = zip_file_object.namelist()[0]
    source = zip_file_object.open(first_file)
else:
    source = cache

df = pd.read_csv(source, sep=";", index_col=0, parse_dates=True, decimal=",")
df.to_csv("LD2011_2014_extrait.txt", sep=";", decimal=",")

In [2]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [3]:
df = pd.read_csv(
    "LD2011_2014_extrait.txt", sep=";", decimal=",", index_col=0, parse_dates=True
)
df = df.astype("float32")

# The raw columns are average power in kW over each 15-min interval. Divided by 4 they become
# the energy consumed during that interval, in kWh -- what the client is actually billed for,
# and the only form in which a daily total is a plain sum.
df = df / 4
df.head()

,MT_001,MT_002,MT_003,MT_004,MT_005,MT_006,MT_007,MT_008,MT_009,MT_010,...,MT_361,MT_362,MT_363,MT_364,MT_365,MT_366,MT_367,MT_368,MT_369,MT_370
2011-01-01 00:15:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2011-01-01 00:30:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2011-01-01 00:45:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2011-01-01 01:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2011-01-01 01:15:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 1. What is actually in the file?

Before any modelling: how many rows, over what period, and is the time grid regular?

In [4]:
print("shape          :", df.shape)
print("period         :", df.index.min(), "->", df.index.max())
print("memory         : %.0f MB" % (df.memory_usage(deep=True).sum() / 1e6))
print("missing values :", int(df.isna().sum().sum()))
df.describe().T.head()

shape          : (140256, 370)
period         : 2011-01-01 00:15:00 -> 2015-01-01 00:00:00
memory         : 209 MB
missing values : 0


,count,mean,std,min,25%,50%,75%,max
MT_001,140256.0,0.992696,1.495991,0.0,0.000000,0.317259,0.634518,12.055838
MT_002,140256.0,5.192120,3.318104,0.0,0.711238,6.223329,7.467994,28.805120
MT_003,140256.0,0.729577,2.753614,0.0,0.000000,0.434405,0.434405,37.793224
MT_004,140256.0,20.546124,14.562099,0.0,9.146341,21.849594,28.963415,80.284554
MT_005,140256.0,9.310078,6.615332,0.0,3.963415,9.756098,13.719512,37.500000


In [5]:
# Is the 15-minute grid regular? Anything other than one value here is a trap for lag features:
# a lag of 96 rows is only "yesterday" if every row is 15 minutes apart.
print(df.index.to_series().diff().value_counts())

intervals_per_day = df.index.to_series().groupby(df.index.date).size()
print()
print("days that do NOT have 96 intervals:")
print(intervals_per_day[intervals_per_day != 96])

0 days 00:15:00    140255
Name: count, dtype: int64

days that do NOT have 96 intervals:
2011-01-01    95
2015-01-01     1
dtype: int64


The **row spacing** is perfectly regular: one 15-minute step everywhere, no gaps. So
`shift(96)` lands on the same clock time yesterday. That is necessary but, as the next cell
shows, not sufficient for the data to be sound.

Two boundary rows to be careful with:

- **2011-01-01 has 95 intervals**, because the series starts at 00:15 and not at 00:00;
- **there is a single row dated 2015-01-01 00:00:00**, which is the last interval of 2014
  labelled with the following midnight.

That second one matters: filtering `year == 2014` silently drops it, and filtering
`year == 2015` yields a one-row dataset. Worth remembering when the split is written.

In [6]:
# The index is regular, but is the CLOCK regular? These timestamps are local time, and Portugal
# changes clock twice a year. Look at the two transition days of 2014.
#
# "The fleet" here and below = all 370 clients taken together, i.e. `df.mean(axis=1)`: the mean
# ACROSS the client columns, one value per 15-min timestamp.
for day, label in [("2014-03-30", "spring forward"), ("2014-10-26", "fall back")]:
    one_day = df.loc[day]
    by_hour = one_day.mean(axis=1).groupby(one_day.index.hour).mean()
    print("%s (%s): %d intervals" % (day, label, len(one_day)))
    print("   fleet mean, hours 00-03 :", [round(v, 1) for v in by_hour.head(4)])

days = [("2014-03-30", "spring forward"), ("2014-10-26", "fall back")]
fig = make_subplots(rows=1, cols=2, shared_yaxes=True,
                    subplot_titles=[f"{label} ({day})" for day, label in days])
for col, (day, label) in enumerate(days, start=1):
    one_day = df.loc[day].mean(axis=1)
    reference = df.loc[pd.Timestamp(day) - pd.Timedelta("7D"):
                       pd.Timestamp(day) - pd.Timedelta("7D") + pd.Timedelta("23:45:00")].mean(axis=1)
    fig.add_scatter(x=one_day.index.hour + one_day.index.minute / 60, y=one_day.values,
                    name=day, line={"color": "firebrick", "width": 2}, row=1, col=col)
    fig.add_scatter(x=reference.index.hour + reference.index.minute / 60, y=reference.values,
                    name="same weekday, 7 days earlier", showlegend=col == 1,
                    line={"color": "dimgray", "width": 1.2, "dash": "dash"}, row=1, col=col)
    fig.add_vrect(x0=1, x1=2, fillcolor="orange", opacity=0.2, line_width=0, row=1, col=col)
    fig.update_xaxes(title_text="hour of day", row=1, col=col)
fig.update_yaxes(title_text="kWh / 15 min", row=1, col=1)
fig.update_layout(template="plotly_white", height=400)
fig.show()


2014-03-30 (spring forward): 96 intervals
   fleet mean, hours 00-03 : [101.7, 0.3, 84.7, 80.8]
2014-10-26 (fall back): 96 intervals
   fleet mean, hours 00-03 : [106.1, 180.7, 86.9, 82.5]


/tmp/ipykernel_84450/2488747639.py:17: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  reference = df.loc[pd.Timestamp(day) - pd.Timedelta("7D"):
/tmp/ipykernel_84450/2488747639.py:18: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  pd.Timestamp(day) - pd.Timedelta("7D") + pd.Timedelta("23:45:00")].mean(axis=1)
/tmp/ipykernel_84450/2488747639.py:17: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  reference = df.loc[pd.Timestamp(day) - pd.Timedelta("7D"):
/tmp/ipykernel_84450/2488747639.py:18: D

**The clock is not regular, and the file hides it.**

The dataset keeps a fixed 96-row grid in *local* time, so the two daylight-saving transitions are
folded into it rather than represented:

- **spring forward**: the hour that never happened locally is still present as four rows, reading
  essentially **zero** against a normal day;
- **fall back**: the hour that happened twice is folded into one labelled hour, which therefore
  reads roughly **double**.

No missing-value check catches this - the rows are there and the numbers are valid floats. It
matters twice over. These days show up as fake anomalies in any daily aggregate, and worse, a
`lag_1d` feature on the day after the spring transition reads that near-zero hour and hands the
model a value that never existed.

Two defensible options: drop the two transition days from training every year, or convert the
index to UTC before building any lag. Doing neither is a choice too - just one that should be
made knowingly.


## 2. Are the 370 clients comparable?

An average over 370 clients only means something if they are of comparable size.

In [7]:
totals = df.sum().sort_values(ascending=False)

fig = go.Figure(go.Bar(x=list(range(len(totals))), y=totals.values / 1e6,
                       customdata=totals.index, marker_color="slategray",
                       hovertemplate="%{customdata}<br>%{y:.2f} GWh<extra></extra>"))
fig.update_yaxes(type="log", title_text="GWh")
fig.update_xaxes(title_text="clients, ranked")
fig.update_layout(template="plotly_white", height=400,
                  title="Total consumption over 2011-2014, one bar per client (log scale)")
fig.show()

print("smallest client : %12.0f kWh" % totals.min())
print("median client   : %12.0f kWh" % totals.median())
print("largest client  : %12.0f kWh  (%.0fx the median)" % (totals.max(), totals.max() / totals.median()))
share = totals.cumsum() / totals.sum()
print()
print("top 10 clients carry %.1f%% of all consumption" % (100 * share.iloc[9]))
print("top 50 clients carry %.1f%% of all consumption" % (100 * share.iloc[49]))


smallest client :        28684 kWh
median client   :      3741130 kWh
largest client  :   1318686464 kWh  (352x the median)

top 10 clients carry 52.5% of all consumption
top 50 clients carry 77.4% of all consumption


The fleet is extremely unequal: the largest client consumes orders of magnitude more than the
median, and a handful of clients account for over half of everything. Note the log scale - on a
linear axis the smallest clients would be invisible.

Consequence for later: a pooled RMSE over all clients is essentially a measure of how well those
few are predicted. Keep that in mind when reading any single error number.


In [8]:
# Zeros are suspicious in consumption data: a meter that reads exactly 0 for months is a meter
# that was not installed yet, not a client consuming nothing.
zero_share = (df == 0).mean()
first_reading = (df != 0).idxmax()

print("clients with at least one zero :", int((zero_share > 0).sum()), "/", df.shape[1])
print("clients that are >50%% zeros     :", int((zero_share > 0.5).sum()))
print()
print("first non-zero reading, distribution by year:")
print(first_reading.dt.year.value_counts().sort_index())

clients with at least one zero : 280 / 370
clients that are >50%% zeros     : 43

first non-zero reading, distribution by year:
2011    160
2012    173
2013     18
2014     19
Name: count, dtype: int64


In [9]:
# So the fleet grows over time. Does that distort a naive year-over-year comparison?
all_clients = df.groupby(df.index.year).mean().mean(axis=1)

complete = df.columns[(df.loc[df.index.year == 2011] != 0).all()]
subset = df[complete].groupby(df.index.year).mean().mean(axis=1)

comparison = pd.DataFrame(
    {
        "all 370 clients": all_clients,
        f"the {len(complete)} clients present in 2011": subset,
    }
).loc[2011:2014].round(2)
print(comparison)
print()
print("all clients      2011 -> 2012 : %+.1f%%" % (100 * (all_clients[2012] / all_clients[2011] - 1)))
print("complete history 2011 -> 2012 : %+.1f%%" % (100 * (subset[2012] / subset[2011] - 1)))

      all 370 clients  the 154 clients present in 2011
2011        82.919998                       197.009995
2012       143.460007                       176.309998
2013       150.500000                       167.520004
2014       151.630005                       165.919998

all clients      2011 -> 2012 : +73.0%
complete history 2011 -> 2012 : -10.5%


**The training window straddles a meter rollout.**

Taken over all clients, mean consumption appears to *grow* sharply between 2011 and 2012.
Restricted to the clients that were already reporting in 2011, it *falls*, and keeps declining
gently through 2014.

The apparent growth is not behaviour, it is **composition**: most meters come online after
mid-2011, and their leading zeros drag the 2011 average down. A model - or a drift alarm - built
on the naive comparison would be chasing an artifact.

Two practical consequences: 2011 is not comparable to the later years, and per-client
normalisation matters more than any fleet-level aggregate.

## 3. What shape does consumption have in time?

Three scales to look at: the year, the week, the day.

In [10]:
fleet_daily = df.mean(axis=1).resample("D").mean()

fig = go.Figure()
for year in (2011, 2012, 2013, 2014):
    one_year = fleet_daily[fleet_daily.index.year == year]
    fig.add_scatter(x=one_year.index.dayofyear, y=one_year.values, name=str(year),
                    line={"width": 1.2})
fig.update_xaxes(title_text="day of year")
fig.update_yaxes(title_text="kWh / 15 min")
fig.update_layout(template="plotly_white", height=450,
                  title="Mean consumption per 15-min interval, daily average, years overlaid")
fig.show()


Overlaying the years shows a clear **annual seasonality** - a summer peak and a winter
shoulder - repeating with the same shape every year. It also shows 2011 sitting well below the
others, which we now know is the meter-rollout artifact and not a real trend.

The sharp isolated dips are worth chasing: a seasonal curve does not fall off a cliff for one
day.

In [11]:
year_2014 = df.loc[df.index.year == 2014]
fleet_2014 = year_2014.mean(axis=1)

by_hour = fleet_2014.groupby(fleet_2014.index.hour).mean()
by_weekday = fleet_2014.groupby(fleet_2014.index.dayofweek).mean()
weekdays = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

fig = make_subplots(rows=1, cols=2, subplot_titles=["Average day (2014)", "Average week (2014)"])
fig.add_scatter(x=by_hour.index, y=by_hour.values, mode="lines+markers",
                line={"color": "#3b6ea5"}, name="by hour", row=1, col=1)
fig.add_bar(x=weekdays, y=by_weekday.values, marker_color="#3b6ea5", name="by weekday",
            row=1, col=2)
fig.update_xaxes(title_text="hour", row=1, col=1)
fig.update_yaxes(title_text="kWh / 15 min", rangemode="tozero")
fig.update_layout(template="plotly_white", height=400, showlegend=False)
fig.show()

print("daily   : min %.1f at %dh, max %.1f at %dh  -> peak/trough = %.2f" % (
    by_hour.min(), by_hour.idxmin(), by_hour.max(), by_hour.idxmax(), by_hour.max() / by_hour.min()))
print("weekly  : weekend / weekday = %.3f" % (
    by_weekday[[5, 6]].mean() / by_weekday[[0, 1, 2, 3, 4]].mean()))


daily   : min 83.0 at 4h, max 202.6 at 18h  -> peak/trough = 2.44
weekly  : weekend / weekday = 0.997


The **daily cycle is the dominant pattern**: the evening peak is more than twice the small
hours.

The **weekly cycle is absent at fleet level** - weekends land within a fraction of a percent of
weekdays. That is counter-intuitive and useful: adding a `weekday` feature would buy nothing on
the aggregate. The likely reason is cancellation - offices drop on Sunday, homes rise - and
individual clients will behave very differently. Worth re-checking per client before dismissing
it for good.


In [12]:
# Chase the dips: December 2014, one client against the fleet.
client = "MT_222"
december = df.loc["2014-12-01":"2014-12-31"]
client_daily = december[client].resample("D").sum()
fleet_daily_dec = december.mean(axis=1).resample("D").sum()

colors = ["firebrick" if d == pd.Timestamp("2014-12-25") else "seagreen" for d in client_daily.index]

fig = go.Figure()
fig.add_bar(x=client_daily.index, y=client_daily.values, marker_color=colors,
            name=f"{client} - daily total (kWh)",
            hovertemplate="%{x|%d %b}<br>%{y:.0f} kWh<extra></extra>")
fig.add_scatter(x=fleet_daily_dec.index, y=fleet_daily_dec.values,
                name="fleet average - daily total (kWh)",
                line={"color": "dimgray", "width": 2, "dash": "dash"})
fig.update_yaxes(title_text="kWh / day")
fig.update_layout(template="plotly_white", height=450,
                  title=f"December 2014: {client} against the fleet average, 25 Dec in red")
fig.show()


In [13]:
# Which days of 2014 are the lowest, fleet-wide?
lowest = fleet_daily[fleet_daily.index.year == 2014].nsmallest(6)
december_median = fleet_daily["2014-12-01":"2014-12-31"].median()
for day, value in lowest.items():
    print("%s  %6.2f kWh/15min  (%3.0f%% of the December median)" % (
        day.date(), value, 100 * value / december_median))

2014-12-25   75.37 kWh/15min  ( 57% of the December median)
2014-01-01   87.09 kWh/15min  ( 65% of the December median)
2014-12-24  121.86 kWh/15min  ( 91% of the December median)
2014-12-31  122.21 kWh/15min  ( 92% of the December median)
2014-02-16  123.81 kWh/15min  ( 93% of the December median)
2014-03-30  123.96 kWh/15min  ( 93% of the December median)


Two different causes sit in that list, and telling them apart matters.

**Public holidays** - Christmas Day is the lowest day of the year, New Year's Day close behind,
with 24 and 31 December visibly reduced. This is real consumption: people genuinely use less, and
the whole fleet drops together.

**The daylight-saving artifact** - the spring transition day is in the list only because of the
near-zero hour found in section 1. That one is not consumption at all.

Two things follow. A holiday calendar would be a genuinely useful feature, because no lag-based
feature can anticipate a holiday - yesterday was an ordinary day. And the two causes need opposite
treatment: holidays are real and must be *predicted*, the DST hour is corrupt and should be
*repaired or dropped*. Lumping both into "outliers" and clipping them would throw away the real
signal and keep the fake one.


## 4. Which past values actually carry information?

Rather than guessing lag windows, measure the autocorrelation.

In [14]:
lags = {
    "15 min": 1, "1 h": 4, "3 h": 12, "6 h": 24, "12 h": 48,
    "1 day": 96, "2 days": 192, "1 week": 672, "30 days": 2880,
}
print("autocorrelation of the fleet mean (2014):")
for label, lag in lags.items():
    print("  %-8s %+.4f" % (label, fleet_2014.autocorr(lag)))

profile = [fleet_2014.autocorr(lag) for lag in range(1, 96 * 8)]
fig = go.Figure(go.Scatter(x=np.arange(1, 96 * 8) / 96, y=profile,
                           line={"color": "#3b6ea5", "width": 1.5},
                           hovertemplate="lag %{x:.2f} days<br>r %{y:.3f}<extra></extra>"))
fig.add_hline(y=0, line={"color": "black", "width": 1})
for day in range(1, 8):
    fig.add_vline(x=day, line={"color": "firebrick", "width": 1, "dash": "dot"})
fig.update_xaxes(title_text="lag in days")
fig.update_yaxes(title_text="correlation")
fig.update_layout(template="plotly_white", height=400, showlegend=False,
                  title="Autocorrelation, 8 days (red lines = exact multiples of 24 h)")
fig.show()


autocorrelation of the fleet mean (2014):
  15 min   +0.9934
  1 h      +0.9515
  3 h      +0.6999
  6 h      +0.1427
  12 h     -0.5740
  1 day    +0.9850
  2 days   +0.9768
  1 week   +0.9643
  30 days  +0.9323


The autocorrelation profile is the empirical justification for every lag we will use:

- the peaks land exactly on **multiples of 24 h**, one day and one week ahead of the rest, which
  is why lags are counted in steps of 96;
- **twelve hours is anti-correlated** - it is the opposite side of the day/night cycle, so a
  "12 h ago" feature would be actively misleading if used naively;
- the decay between the daily peaks is shallow, so `lag_1d` and `lag_7d` carry most of the
  signal and a longer `lag_30d` adds smoothed level rather than shape.


In [15]:
# Candidate features for EVERY client, stacked long, with the client kept as a column.
frames = []
for name in df.columns:
    one = df[[name]].copy()
    one.columns = ["consumption"]
    one["lag_1d"] = one["consumption"].shift(96)
    one["lag_7d"] = one["consumption"].shift(96 * 7)
    one["lag_30d"] = one["consumption"].shift(96 * 30)
    one["lag_365d"] = one["consumption"].shift(96 * 365)
    # The rolling mean is computed on the SHIFTED series: including the current value would let
    # the target predict itself.
    one["rolling_mean_7d"] = one["consumption"].shift(1).rolling(96 * 7).mean().astype("float32")
    one["rolling_mean_30d"] = one["consumption"].shift(1).rolling(96 * 30).mean().astype("float32")
    one["client"] = name
    frames.append(one)

tmp = pd.concat(frames).sort_index()
tmp["client"] = tmp["client"].astype("category")
print("rows:", tmp.shape[0], "  clients:", tmp["client"].nunique())

correlations = tmp.corr(numeric_only=True)["consumption"].drop("consumption").sort_values(ascending=False)
print(correlations.round(3))
print()
print("rows lost to dropna, per feature set:")
print("  lag_1d only            : %10d" % tmp.dropna(subset=["lag_1d"]).shape[0])
print("  + lag_7d, lag_30d      : %10d" % tmp.dropna(subset=["lag_1d", "lag_7d", "lag_30d"]).shape[0])
print("  + lag_365d             : %10d" % tmp.dropna(subset=["lag_1d", "lag_7d", "lag_30d", "lag_365d"]).shape[0])
print("  total rows             : %10d" % tmp.shape[0])
print()
# The aggregate is one answer; individual clients are another. Largest, median, smallest.
totals = df.sum().sort_values(ascending=False)
clients = [totals.index[0], totals.index[len(totals) // 2], totals.index[-1]]
print("per client (%s):" % ", ".join(clients))
print(pd.DataFrame({
    name: tmp[tmp["client"] == name].corr(numeric_only=True)["consumption"].drop("consumption")
    for name in clients
}).round(3))


rows: 51894720   clients: 370
lag_1d              0.991
lag_7d              0.980
lag_30d             0.955
rolling_mean_7d     0.881
rolling_mean_30d    0.875
lag_365d            0.797
Name: consumption, dtype: float64

rows lost to dropna, per feature set:
  lag_1d only            :   51859200
  + lag_7d, lag_30d      :   50829120
  + lag_365d             :   38929920
  total rows             :   51894720

per client (MT_362, MT_320, MT_133):
                  MT_362  MT_320  MT_133
lag_1d             0.983   0.911   0.783
lag_7d             0.956   0.901   0.710
lag_30d            0.899   0.859   0.626
lag_365d           0.535   0.804     NaN
rolling_mean_7d    0.692   0.309   0.744
rolling_mean_30d   0.670   0.293   0.717


In [ ]:
matrix = tmp.corr(numeric_only=True)
labels = list(matrix.columns)

fig = go.Figure(go.Heatmap(z=matrix.values, x=labels, y=labels, zmin=0, zmax=1,
                           colorscale=[(0.0, "#cde2fb"), (0.15, "#9ec5f4"), (0.3, "#6da7ec"), (0.45, "#3987e5"),
              (0.6, "#2a78d6"), (0.75, "#256abf"), (0.9, "#184f95"), (1.0, "#0d366b")],
                           xgap=2, ygap=2, texttemplate="%{z:.2f}", textfont={"size": 11},
                           hovertemplate="%{y} / %{x}<br>correlation %{z:.3f}<extra></extra>",
                           colorbar={"title": "correlation", "thickness": 14}))
fig.update_layout(template="plotly_white", width=720, height=620,
                  title="Feature correlations, all 370 clients",
                  margin={"l": 120, "r": 40, "t": 60, "b": 120})
fig.update_yaxes(autorange="reversed")
fig.show()


Over all clients `lag_1d` leads, and the ranking per client is **not** the same: on a
weekday-driven client `lag_7d` overtakes it, because that client resembles the same weekday last
week far more than it resembles yesterday. Nothing about "one day ago" is universally best - the
aggregate describes the biggest meters, and any single client can disagree with it.

`lag_365d` is the worst trade on the board: middling correlation and it costs a quarter of all
rows to the `dropna`, because a year of history has to exist before the first usable row. Note
also that the drop is per *strategy* - dropping rows for a feature the strategy does not use would
be paying the price for nothing.

The features are also heavily correlated with each other, being lags of the same series. That is
the situation where an unpenalised linear fit gets unstable coefficients, and the reason to reach
for Ridge.


## 5. The bar to beat

Before fitting anything: how good is "same time yesterday"? A model that cannot beat that is
not worth deploying.

In [16]:
window = df.loc["2013-12-01":"2014-12-31"]
is_2014 = window.index.year == 2014

for label, lag in [("yesterday, same time", 96), ("last week, same time", 672)]:
    errors = (window[is_2014] - window.shift(lag)[is_2014]).to_numpy().ravel()
    errors = errors[~np.isnan(errors)]
    rmse = np.sqrt((errors ** 2).mean())
    mae = np.abs(errors).mean()
    print("%-22s RMSE = %7.2f kWh   MAE = %6.2f kWh" % (label, rmse, mae))

yesterday, same time   RMSE =  117.46 kWh   MAE =  14.22 kWh
last week, same time   RMSE =  173.30 kWh   MAE =  16.54 kWh


Persistence at one day is the bar to beat over the whole fleet: a model scoring worse than the
naive "same time yesterday" is worse than doing nothing.

Note the gap between RMSE and MAE. The squared error is dominated by a handful of very large
clients - consistent with the concentration measured in section 2 - so a single pooled RMSE hides
most of what is going on.


## 6. A first model

Chronological split - never a random one on a time series, or the model gets to see its own
future.

In [25]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

data = tmp.dropna(subset=["lag_1d", "lag_7d", "lag_30d", "rolling_mean_30d"]).copy()
features = ["lag_1d", "lag_7d", "lag_30d", "rolling_mean_30d"]

# Chronological on the stacked frame: every client's rows sit under the same timestamps, so the
# cut is a moment in time, not a client boundary.
n_train = int(len(data) * 0.8)
train, test = data.iloc[:n_train], data.iloc[n_train:]
Xtr, ytr = train[features], train["consumption"]
Xte, yte = test[features], test["consumption"]
print("train:", Xtr.shape, train.index.min().date(), "->", train.index.max().date())
print("test :", Xte.shape, test.index.min().date(), "->", test.index.max().date())

lr = LinearRegression()
lr.fit(Xtr, ytr)
print()
print("train RMSE %.3f | test RMSE %.3f" % (
    mean_squared_error(ytr, lr.predict(Xtr)) ** 0.5,
    mean_squared_error(yte, lr.predict(Xte)) ** 0.5))


train: (40663296, 4) 2011-01-31 -> 2014-03-20
test : (10165824, 4) 2014-03-20 -> 2015-01-01

train RMSE 99.479 | test RMSE 118.083


In [35]:
# try a couple of alphas
m = Ridge(alpha=1.0)
m.fit(Xtr, ytr)
mean_squared_error(yte, m.predict(Xte)) ** 0.5

118.08057613738806

In [32]:
m = Ridge(alpha=1000.0)
m.fit(Xtr, ytr)
mean_squared_error(yte, m.predict(Xte)) ** 0.5

123.78963617135928

In [33]:
# maybe fewer features generalize better? try short memory only
features = ["lag_1d"]
Xtr, ytr = train[features], train["consumption"]
Xte, yte = test[features], test["consumption"]
m = Ridge(alpha=1.0)
m.fit(Xtr, ytr)
mean_squared_error(yte, m.predict(Xte)) ** 0.5

123.78963617135928

## 7. Is the model any good?

A single RMSE says almost nothing. Look at where the error lives.

In [34]:
# Refit on the four features so the diagnostics below match the model being judged.
features = ["lag_1d", "lag_7d", "lag_30d", "rolling_mean_30d"]
Xtr, ytr = train[features], train["consumption"]
Xte, yte = test[features], test["consumption"]
model = Ridge(alpha=1.0).fit(Xtr, ytr)
predicted = pd.Series(model.predict(Xte), index=Xte.index)
residuals = yte - predicted

print("test RMSE %.3f | MAE %.3f | bias (mean residual) %+.3f" % (
    mean_squared_error(yte, predicted) ** 0.5,
    mean_absolute_error(yte, predicted),
    residuals.mean()))

test RMSE 118.081 | MAE 13.622 | bias (mean residual) -0.138


In [22]:
# One day, actual against predicted, over all clients and then for two of them.
# An ordinary Wednesday on purpose: a holiday or a DST transition would show the data's problems
# rather than the model's.
day = pd.Timestamp("2014-06-11")
window_day = slice(day, day + pd.Timedelta("23:45:00"))
panels = ["all clients (mean)"] + [str(name) for name in clients[:2]]

fig = make_subplots(rows=1, cols=len(panels), subplot_titles=panels, shared_yaxes=False)
for col, panel in enumerate(panels, start=1):
    subset = test.loc[window_day]
    if col > 1:
        subset = subset[subset["client"] == panel]
    curves = subset.assign(predicted=model.predict(subset[features])).groupby(
        level=0).mean(numeric_only=True)
    for name, values, color, dash in [
        ("actual", curves["consumption"], "seagreen", "solid"),
        ("Ridge prediction", curves["predicted"], "orange", "solid"),
        ("yesterday (baseline)", curves["lag_1d"], "#3b6ea5", "dot"),
    ]:
        fig.add_scatter(x=curves.index, y=values, name=name, showlegend=col == 1,
                        line={"color": color, "dash": dash, "width": 2}, row=1, col=col)
fig.update_yaxes(title_text="kWh / 15 min", col=1)
fig.update_layout(template="plotly_white", height=420,
                  title=f"{day.date()}: actual vs prediction")
fig.show()


/tmp/ipykernel_84450/2143254829.py:5: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  window_day = slice(day, day + pd.Timedelta("23:45:00"))


In [23]:
# 10M residuals: bin them before plotting instead of shipping every point to the browser.
by_hour_error = residuals.groupby(residuals.index.hour).apply(lambda s: (s ** 2).mean() ** 0.5)
counts, prediction_edges, residual_edges = np.histogram2d(predicted.values, residuals.values, bins=120)
distribution, residual_bins = np.histogram(residuals.values, bins=80)

fig = make_subplots(rows=1, cols=3, subplot_titles=["Residual vs predicted",
                                                    "Residual distribution",
                                                    "RMSE by hour of day"])
fig.add_heatmap(x=prediction_edges[:-1], y=residual_edges[:-1], z=np.log10(counts.T + 1),
                colorscale="Blues", colorbar={"title": "log10 rows", "thickness": 12, "x": 0.28},
                row=1, col=1)
fig.add_bar(x=residual_bins[:-1], y=distribution, marker_color="#3b6ea5", row=1, col=2)
fig.add_bar(x=by_hour_error.index, y=by_hour_error.values, marker_color="#3b6ea5", row=1, col=3)
fig.add_hline(y=0, line={"color": "black", "width": 1}, row=1, col=1)
fig.update_xaxes(title_text="predicted (kWh)", row=1, col=1)
fig.update_yaxes(title_text="residual (kWh)", row=1, col=1)
fig.update_xaxes(title_text="residual (kWh)", row=1, col=2)
fig.update_xaxes(title_text="hour", row=1, col=3)
fig.update_yaxes(title_text="RMSE (kWh)", row=1, col=3)
fig.update_layout(template="plotly_white", height=400, showlegend=False)
fig.show()


What the three panels say, over the **whole test set** (10.2M rows, binned rather than sampled):

- **Residual vs predicted** should be a shapeless band around zero. The density is concentrated
  near the origin because most intervals are small; the tail to the right is the handful of very
  large clients, and that is where the pooled RMSE comes from.
- **The residual distribution** is sharply peaked with long tails: most intervals are predicted
  well and a few are badly wrong. That is the RMSE-versus-MAE gap, now visible.
- **RMSE by hour is far from flat**: the error concentrates in the working hours, when
  consumption moves fastest. That is a direct argument for an explicit `hour` feature - which
  the current feature set does not have.


In [24]:
# Where does the pooled error come from? First by consumption level, then by client.
buckets = pd.qcut(yte, 10, labels=False, duplicates="drop")
per_bucket = pd.DataFrame({"actual": yte.to_numpy(), "residual": residuals.to_numpy(),
                           "bucket": buckets.to_numpy()})
summary = per_bucket.groupby("bucket").apply(
    lambda g: pd.Series({
        "mean actual": g["actual"].mean(),
        "rmse": (g["residual"] ** 2).mean() ** 0.5,
        "rmse % of level": 100 * (g["residual"] ** 2).mean() ** 0.5 / max(g["actual"].mean(), 1e-9),
    }),
    include_groups=False,
)
print(summary.round(2))
print()

per_client_error = (test.assign(error=residuals.to_numpy())
                    .groupby("client", observed=True)
                    .apply(lambda g: (g["error"] ** 2).mean() ** 0.5, include_groups=False)
                    .sort_values(ascending=False))
print("pooled RMSE %.2f | worst client %.2f | median client %.2f | best client %.2f" % (
    (residuals ** 2).mean() ** 0.5, per_client_error.max(),
    per_client_error.median(), per_client_error.min()))
print(per_client_error.head(3).round(2).to_dict())


        mean actual        rmse  rmse % of level
bucket                                          
0          2.570000   39.349998      1529.930054
1          7.830000    3.640000        46.419998
2         12.600000    5.610000        44.509998
3         19.030001    4.270000        22.430000
4         25.889999   11.420000        44.119999
5         35.250000    6.710000        19.030001
6         49.990002    9.320000        18.629999
7         78.970001   12.660000        16.030001
8        155.699997   23.959999        15.390000
9       1186.660034  369.920013        31.170000

pooled RMSE 118.08 | worst client 1975.90 | median client 4.79 | best client 0.32
{'MT_362': 1975.9000244140625, 'MT_370': 787.010009765625, 'MT_196': 566.219970703125}


Two readings, and they point in opposite directions.

**By consumption level**, the relative error falls steadily across the middle deciles -
proportionally the model is at its best on the large intervals. The two ends are the exception:
the lowest decile is hopeless in relative terms, and the top decile pays in absolute terms.

**By client**, the pooled RMSE covers a range of several orders of magnitude, and the worst
clients are exactly the large meters found in section 2. The median client scores far better than
the number usually quoted as "the model's accuracy".

A single pooled RMSE mixes all of this and mostly reports the largest clients. For a per-client
service a relative metric - MAPE, or RMSE normalised by the client's own mean - describes quality
far better.


## 8. Does it overfit?

The question is not "is the training error low" but "is the gap between training and unseen
data acceptable".

In [ ]:
from sklearn.tree import DecisionTreeRegressor

# An unbounded tree stores one leaf per training row, which 40M rows will not survive, so this
# comparison runs on a 1% row sample. Rows are sampled - every client stays in.
sample = data.sample(frac=0.01, random_state=0).sort_index()
n_sample_train = int(len(sample) * 0.8)
sample_train, sample_test = sample.iloc[:n_sample_train], sample.iloc[n_sample_train:]
Xtr_s, ytr_s = sample_train[features], sample_train["consumption"]
Xte_s, yte_s = sample_test[features], sample_test["consumption"]

candidates = {
    "Ridge(alpha=1)": Ridge(alpha=1.0),
    "DecisionTree(max_depth=6)": DecisionTreeRegressor(max_depth=6, random_state=0),
    "DecisionTree(no limit)": DecisionTreeRegressor(random_state=0),
}
rows = []
for name, candidate in candidates.items():
    candidate.fit(Xtr_s, ytr_s)
    rows.append({
        "model": name,
        "train RMSE": mean_squared_error(ytr_s, candidate.predict(Xtr_s)) ** 0.5,
        "test RMSE": mean_squared_error(yte_s, candidate.predict(Xte_s)) ** 0.5,
    })
overfitting = pd.DataFrame(rows).set_index("model")
# No ratio column on purpose: the unbounded tree reaches a training error of essentially zero, so
# test/train is a division by zero. The two numbers side by side say it better anyway.
print("sample: %d train rows, %d clients" % (Xtr_s.shape[0], sample["client"].nunique()))
print(overfitting.round(3))


Read the two columns together, not the training one alone. The unbounded tree is the textbook
case: it drives the training error to essentially *zero* by storing a leaf per training row, and
it has by far the worst test error. Same model family, one `max_depth` away from being
competitive.

What matters is the gap, not the level: **a training error near zero is a warning sign, not an
achievement.** Note also that the depth-limited tree loses here where on a single client it used
to win - all clients stacked give a tree far more to memorise, and the pooled test error charges
for it.

In [ ]:
# The regularisation curve: too little and the model chases noise, too much and it predicts a
# constant. The useful alpha is where the validation curve bottoms out.
alphas = [0.0, 1e2, 1e4, 1e6, 1e7, 3e7, 1e8, 3e8, 1e9]
train_curve, test_curve = [], []
for alpha in alphas:
    candidate = Ridge(alpha=alpha).fit(Xtr, ytr)
    train_curve.append(mean_squared_error(ytr, candidate.predict(Xtr)) ** 0.5)
    test_curve.append(mean_squared_error(yte, candidate.predict(Xte)) ** 0.5)

labels = [f"{a:g}" for a in alphas]
fig = go.Figure()
fig.add_scatter(x=labels, y=train_curve, name="train", mode="lines+markers",
                line={"color": "#3b6ea5"})
fig.add_scatter(x=labels, y=test_curve, name="test", mode="lines+markers",
                line={"color": "orange"}, marker={"symbol": "square"})
fig.update_xaxes(title_text="alpha", type="category")
fig.update_yaxes(title_text="RMSE (kWh)")
fig.update_layout(template="plotly_white", height=450, title="Regularisation curve")
fig.show()

best = int(np.argmin(test_curve))
print("lowest test RMSE at alpha = %g (%.3f)" % (alphas[best], test_curve[best]))


The curve is **flat over several orders of magnitude and then goes up**. Both train and test rise
together once alpha is large enough to bite: the coefficients get crushed towards zero and the
model degenerates towards predicting a constant. That is **underfitting** - the opposite failure
from the tree above, and the two are visible in the same notebook.

What is *not* here is an interior optimum: the lowest test RMSE sits at the smallest penalty
tried, so the penalty buys nothing on this data. Worth knowing before spending a day tuning it -
the feature set is the lever.

One caveat to write down, because it bites later: **alpha is not scale-free.** The loss term grows
with the square of the target and with the number of rows, while the penalty does not. So the
useful alpha on the full stacked training set is nothing like the useful alpha on one client - and
converting kW to kWh alone moves it by a factor of 16. Any alpha has to be re-selected when the
data, the row count or the unit changes.


## 9. Save whatever is current

`m` is whatever the last cell left behind.

In [ ]:
import pickle

pickle.dump(m, open("model.pkl", "wb"))